# EDA for Telco Customer Churn dataset

[dataset link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
[dataset description](https://community.ibm.com/community/user/blogs/steven-macko/2019/07/11/telco-customer-churn-1113)

In [ ]:
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

## Download dataset

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.head()

## Prelimitary analys

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
is_na_default = df.isna().sum()
is_na_default.name = "is_na_default"

is_na_space = df.astype(str).map(lambda x: x == "" or x == " ").sum()
is_na_space.name = "is_na_space"

pd.concat([is_na_default, is_na_space], axis = 1)

In [ ]:
df.duplicated().sum()

## Data preparation (basic)

In [ ]:
to_category_columns = (
    [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaymentMethod",
        "TotalCharges",
        ])
    
for columnn in to_category_columns:
    df[columnn] = df[columnn].astype("category")
    

In [ ]:
df.dtypes

In [ ]:
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.describe()

## Data Observation

Check data distribution

In [ ]:
from dataclasses import dataclass
import math

# Utilities for plot generation

@dataclass
class DiscretePlot():
    data: pd.Series
    title: str
    x_title:str | None = None
    y_title: str| None = None
    
def _count_plot_sizes(len_plot_map: int) -> tuple[int, int]:
    columns = 2 if len_plot_map == 2 else 3
    if len_plot_map < columns:
        columns = len_plot_map
    
    rows = math.ceil(len_plot_map / columns)
    return rows, columns
    
def plot_discrete_vars(plot_map: list[DiscretePlot]):
    nrows, ncolumns = _count_plot_sizes(len(plot_map))
    fig, axs = plt.subplots(nrows, ncolumns, figsize=(5 * ncolumns, 4 * nrows))
    
    for i, plot in enumerate(plot_map):
        ax = axs.flat[i] # type: ignore
        
        counts = plot.data.value_counts()
        sns.barplot(x=counts.index, y=counts.values, ax=ax)
        
        ax.set_title(plot.title)
        ax.set_xlabel(plot.x_title if plot.x_title else "")
        ax.set_ylabel(plot.y_title if plot.y_title else "Count")
        ax.tick_params(axis='x', labelrotation=45)
    
    total_cells = nrows * ncolumns
    if total_cells > len(plot_map):
        for j in range(len(plot_map), total_cells):
            axs.flat[j].set_visible(False) # type: ignore
    
    print("Proceeded", len(plot_map)," maps")
    plt.tight_layout()
    plt.show()

In [ ]:
dicrete_plot_map = [
    DiscretePlot(df["gender"], title="Customer gender"),
    DiscretePlot(df["SeniorCitizen"], title="Customer senior citizen", x_title="Are customers senior citizens"),
    DiscretePlot(df["Partner"], title="Customer with partner", x_title="Do customers have a partner"),
    DiscretePlot(df["Dependents"], title="Customer having dependents", x_title="Do customers have a dependent"),
    DiscretePlot(df["PhoneService"], title="Customer with phone service", x_title="Do customers include phone service plan"),
    DiscretePlot(df["MultipleLines"], title="Customer with multiple lines"),
    DiscretePlot(df["InternetService"], title="Customer internet service type"),
    DiscretePlot(df["OnlineSecurity"], title="Customer with online security", x_title="Do customers buy online security"),
    DiscretePlot(df["OnlineBackup"], title="Customer with online backup", x_title="Do customers buy online beckup"),
    DiscretePlot(df["DeviceProtection"], title="Customer with device protection", x_title="Do customers buy device protection"),
    DiscretePlot(df["DeviceProtection"], title="Customer with premium tech support", x_title="Do customers buy premium tech support"),
    DiscretePlot(df["StreamingTV"], title="Customer streaming TV", x_title="Do customers stream TV from another providers"),
    DiscretePlot(df["StreamingMovies"], title="Customer streaming movies", x_title="Do customers stream movies from another providers"),
    DiscretePlot(df["Contract"], title="Customer contract types"),
    DiscretePlot(df["PaperlessBilling"], title="Customer use paperless billing"),
    DiscretePlot(df["PaymentMethod"], title="Customer payment method"),
    DiscretePlot(df["Churn"], title="Customer left", x_title="Do customers left this quarter"),
]

plot_discrete_vars(dicrete_plot_map)

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(25,8))
sns.histplot(df.tenure, bins=20, kde=True, ax=axs[0])
axs[0].set_title("Tenure density")


sns.histplot(df["MonthlyCharges"], bins=20, kde=True, ax=axs[1])
axs[1].set_title("Monthly charges density")


sns.histplot(df["TotalCharges"], bins=20, kde=True)
axs[2].set_title("Total charges density")


plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

def encode_columns(df, columns_list):
    for column in columns_list:
        df[column] = LabelEncoder().fit_transform(df[column])
    return df

In [ ]:
# Encode binary columns with LabelEncoder
binary_columns =(
        [
                "gender", 
                "SeniorCitizen", 
                "Partner", 
                "Dependents", 
                "PhoneService", 
                "PaperlessBilling", 
                "Churn"
        ])

numeric_columns = (
        [
                "tenure",
                "MonthlyCharges",
                "TotalCharges",
        ]
)

df_enc = df.copy()
df_enc = encode_columns(df_enc, binary_columns)


cols = list(df_enc.select_dtypes('number').columns)
corr = df_enc[cols].corr()

plt.figure(figsize=(10, 8))
ax = sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                 center=0, vmin=-1, vmax=1, square=True)

target = 'Churn'
for i, column in enumerate(ax.get_xticklabels()):
    if column.get_text() == target:
        column.set_color('darkred')
        column.set_fontweight('bold')
        column.set_fontsize(20)
for i, column in enumerate(ax.get_yticklabels()):
    if column.get_text() == target:
        column.set_color('darkred')
        column.set_fontweight('bold')
        column.set_fontsize(20)

plt.title("Correlation heatmap (Churn is a target)")
plt.show()

See many small correlations with target. And many multicoleniarities (do not cover category labels yet btw). Sees to better to use stable on multicolearity models

In [ ]:
len(df.columns) - 1

In [ ]:
def plotPointPlots(column_map: list[str]):
    nrows, ncolumns = _count_plot_sizes(len(column_map))
    fig, axs = plt.subplots(nrows, ncolumns, figsize=(5 * ncolumns, 4 * nrows))
    
    for i, column in enumerate(column_map):
        ax = axs.flat[i] # type: ignore
        
        sns.pointplot(df_enc, x= column, y = "Churn", estimator = "mean", ax=ax)
        
        # ax.set_title(column)
        ax.tick_params(axis='x', labelrotation=45)
    
    total_cells = nrows * ncolumns
    if total_cells > len(column_map):
        for j in range(len(column_map), total_cells):
            axs.flat[j].set_visible(False) # type: ignore
    
    print("Proceeded", len(column_map)," maps")
    plt.tight_layout()
    plt.show()

In [ ]:
columns_to_exclude = list(df.select_dtypes('number').columns) + ["customerID", "Churn"]

point_columns = set(df.columns) - set(columns_to_exclude)

plotPointPlots(list(point_columns))

In [ ]:
pairplot_fig = sns.pairplot(df, kind="kde",hue="Churn")
pairplot_fig.fig.suptitle("Data KDE pairplot", y = 1.02)
plt.show()